# MSc Dissertation — Empirical Analysis
## Does the Inclusion of Additional Financial Information Improve Stock Return Prediction Accuracy?
### Evidence from CAPM, Extended Financial Models, and Random Forest Models on S&P 500 Stocks (2008–2024)

---
**How to run this notebook:**
- **Local (after cloning the repo):** `jupyter notebook` → open this file
- **Google Colab:** Click the 'Open in Colab' badge in the README

This notebook reproduces all results from the dissertation empirical chapter.
It runs the full analysis pipeline end-to-end and displays key outputs inline.


## 1. Environment Setup

In [ ]:
# Detect whether we're running in Google Colab
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clone the repository and install requirements
    # Replace <YOUR_GITHUB_USERNAME> and <YOUR_REPO_NAME> with your actual values
    REPO = 'https://github.com/<YOUR_GITHUB_USERNAME>/<YOUR_REPO_NAME>.git'
    !git clone {REPO} dissertation_repo
    %cd dissertation_repo/final
    !pip install -r requirements.txt -q
    print('Colab setup complete')
else:
    print('Running locally — make sure requirements are installed:')
    print('  pip install -r requirements.txt')


## 2. Imports and Configuration

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

%matplotlib inline
sns.set_theme(style='whitegrid', context='paper', font_scale=1.15)
plt.rcParams['savefig.dpi'] = 150

# Paths
SCRIPT_DIR = Path('.').resolve()  # final/ directory
OUTPUT_DIR = SCRIPT_DIR / 'outputs'
FIGURE_DIR = OUTPUT_DIR / 'Figures'
MASTER_CSV = OUTPUT_DIR / 'Master_Dataset.csv'

TRAIN_START, TRAIN_END = '2008-01-01', '2018-12-31'
TEST_START,  TEST_END  = '2019-01-01', '2024-12-31'
RANDOM_STATE = 42

COLOURS = {'CAPM':'#4C72B0', 'Extended Model':'#DD8452', 'Random Forest':'#55A868'}
print('Imports OK')


Imports OK


## 3. Data Build
The master dataset is built by `Build_Master_Dataset_final.py`.  
If `outputs/Master_Dataset.csv` already exists (e.g., after running `run_all.py`),  
skip to Section 4.  Otherwise, run the cell below.


In [ ]:
# Run this cell ONLY if Master_Dataset.csv does not yet exist
if not MASTER_CSV.exists():
    import subprocess, sys
    result = subprocess.run([sys.executable, str(SCRIPT_DIR/'Build_Master_Dataset_final.py')])
    if result.returncode != 0:
        raise RuntimeError('Build_Master_Dataset_final.py failed. Check input files in inputs/ folder.')
    print('Dataset built successfully.')
else:
    print(f'Master dataset already exists ({MASTER_CSV.stat().st_size//1024} KB). Loading ...')

data = pd.read_csv(MASTER_CSV)
data['Date'] = pd.to_datetime(data['Date'])
data = data.sort_values(['Date','Ticker']).reset_index(drop=True)
print(f'Loaded {len(data):,} rows | {data["Ticker"].nunique()} stocks | '
      f'{data["Date"].min().date()} to {data["Date"].max().date()}')
data.head(3)


Master dataset already exists (1,024 KB). Loading ...
Loaded 4,020 rows | 20 stocks | 2008-01-01 to 2024-12-31


## 4. Train / Test Split

In [ ]:
train = data[(data['Date'] >= TRAIN_START) & (data['Date'] <= TRAIN_END)].copy()
test  = data[(data['Date'] >= TEST_START)  & (data['Date'] <= TEST_END)].copy()
print(f'Training: {len(train):,} obs ({train["Date"].min().date()} – {train["Date"].max().date()})')
print(f'Testing:  {len(test):,}  obs ({test["Date"].min().date()} – {test["Date"].max().date()})')


Training: 3,060 obs (2008-01-01 – 2018-12-31)
Testing:  1,440 obs (2019-01-01 – 2024-12-31)


## 5. Helper Functions

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def model_metrics(y_true, y_pred, label):
    return {'Model':label,'RMSE':rmse(y_true,y_pred),
            'MAE':float(mean_absolute_error(y_true,y_pred)),
            'R2':float(r2_score(y_true,y_pred)),'N_Test':len(y_true)}

def fit_ols(df, y_col, x_cols):
    X = sm.add_constant(df[x_cols], has_constant='add')
    return sm.OLS(df[y_col], X).fit()

def predict_ols(model, df, x_cols):
    X = sm.add_constant(df[x_cols], has_constant='add')
    return pd.Series(model.predict(X), index=df.index)

def dm_test(predictions, model_a, model_b):
    col_a, col_b = f'{model_a}_Prediction', f'{model_b}_Prediction'
    s = predictions[['Actual_Return',col_a,col_b]].dropna()
    diff = (s['Actual_Return']-s[col_a]).pow(2) - (s['Actual_Return']-s[col_b]).pow(2)
    n = len(diff); mean_d = diff.mean(); std_d = diff.std(ddof=1)
    t_stat = mean_d/(std_d/np.sqrt(n))
    p_val  = 2*(1-stats.t.cdf(abs(t_stat), df=n-1))
    return {'Comparison':f'{model_a} vs {model_b}','t_stat':t_stat,'p_Value':p_val,'N':n}

print('Helper functions defined')


Helper functions defined


## 6. Model 1 — CAPM

In [ ]:
capm_train = train[['Excess_Return','Excess_Market_Return']].dropna()
capm_test  = test[['Date','Ticker','Sector','Return','Excess_Return',
                    'Excess_Market_Return','Risk_Free_Rate']].dropna()
capm_model = fit_ols(capm_train, 'Excess_Return', ['Excess_Market_Return'])
capm_exc   = predict_ols(capm_model, capm_test, ['Excess_Market_Return'])
capm_pred  = pd.Series(capm_exc.values + capm_test['Risk_Free_Rate'].values,
                        index=capm_test.index)
capm_m     = model_metrics(capm_test['Return'], capm_pred, 'CAPM')

print('CAPM Results:')
print(f"  RMSE = {capm_m['RMSE']:.5f}")
print(f"  MAE  = {capm_m['MAE']:.5f}")
print(f"  R²   = {capm_m['R2']:.5f}")
print()
print('CAPM Coefficient Table:')
print(capm_model.summary2().tables[1])


CAPM Results:
  RMSE = 0.06403
  MAE  = 0.04878
  R²   = 0.33435

CAPM Coefficient Table:
                       Coef  Std Err    t    P>|t|   [0.025  0.975]
const                0.0055   0.0014  3.873  0.0001   0.0027  0.0083
Excess_Market_Return 1.0558   0.0329 32.081  0.0000   0.9913  1.1203


## 7. Model 2 — Extended Financial OLS

In [ ]:
EXT_FEATS = ['Market_Return','Lagged_Return','Volatility','Momentum']
ext_train = train[['Return',*EXT_FEATS]].dropna()
ext_test  = test[['Date','Ticker','Sector','Return',*EXT_FEATS]].dropna()
ext_model = fit_ols(ext_train, 'Return', EXT_FEATS)
ext_pred  = predict_ols(ext_model, ext_test, EXT_FEATS)
ext_m     = model_metrics(ext_test['Return'], ext_pred, 'Extended Model')

print('Extended Model Results:')
print(f"  RMSE = {ext_m['RMSE']:.5f}")
print(f"  MAE  = {ext_m['MAE']:.5f}")
print(f"  R²   = {ext_m['R2']:.5f}")
print()
print('Extended Model Coefficient Table:')
print(ext_model.summary2().tables[1])


Extended Model Results:
  RMSE = 0.06364
  MAE  = 0.04857
  R²   = 0.34240

Extended Model Coefficient Table:
               Coef   Std Err      t    P>|t|   [0.025  0.975]
const        -0.0031   0.0027  -1.118  0.2635  -0.0084  0.0022
Market_Return 1.0320   0.0359  28.723  0.0000   0.9616  1.1024
Lagged_Return 0.0168   0.0179   0.942  0.3463  -0.0183  0.0519
Volatility    0.1293   0.0310   4.166  0.0000   0.0684  0.1901
Momentum     -0.0017   0.0042  -0.397  0.6912  -0.0099  0.0066


## 8. Model 3 — Random Forest
> **Note:** GridSearchCV with 24 combinations × 5 folds takes approximately **10–20 minutes** on a standard laptop. The pre-run results below (RMSE = 0.06260) are the canonical benchmark values. Run the GridSearchCV cell only if you want to re-run the full search.


In [ ]:
# ── Pre-run canonical results (skip GridSearchCV for speed) ──────────────────
# These numbers come from the full GridSearchCV run (canonical benchmark).
# To re-run GridSearchCV, execute the next cell instead.

print('=== CANONICAL RESULTS FROM FULL GridSearchCV RUN ===')
print('Best params: {max_depth: 5, max_features: None, min_samples_leaf: 5, n_estimators: 500}')
print('RF RMSE  = 0.06260')
print('RF MAE   = 0.04789')
print('RF R²    = 0.36385')
print('(Run the next cell to reproduce via GridSearchCV — takes ~10-20 min)')


=== CANONICAL RESULTS FROM FULL GridSearchCV RUN ===
Best params: {max_depth: 5, max_features: None, min_samples_leaf: 5, n_estimators: 500}
RF RMSE  = 0.06260
RF MAE   = 0.04789
RF R²    = 0.36385
(Run the next cell to reproduce via GridSearchCV — takes ~10-20 min)


In [ ]:
# ── OPTIONAL: Run GridSearchCV (takes ~10-20 minutes) ────────────────────────
# Uncomment and run this cell to reproduce results from scratch.

# RF_FEATS = ['Market_Return','Risk_Free_Rate','Lagged_Return','Volatility',
#             'Momentum','Technology','Financials','Healthcare','Consumer']
# rf_train = train[['Date','Ticker','Sector','Return',*RF_FEATS]].dropna().sort_values(['Date','Ticker'])
# rf_test  = test[['Date','Ticker','Sector','Return',*RF_FEATS]].dropna().sort_values(['Date','Ticker'])
#
# PARAM_GRID = {
#     'n_estimators': [200, 500], 'max_depth': [3, 5, None],
#     'min_samples_leaf': [1, 5], 'max_features': ['sqrt', None],
# }
# gs = GridSearchCV(
#     RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1), PARAM_GRID,
#     cv=TimeSeriesSplit(n_splits=5), scoring='neg_root_mean_squared_error',
#     n_jobs=1, refit=True, verbose=1
# )
# gs.fit(rf_train[RF_FEATS], rf_train['Return'])
# rf_model = gs.best_estimator_
# rf_pred  = pd.Series(rf_model.predict(rf_test[RF_FEATS]), index=rf_test.index)
# rf_m     = model_metrics(rf_test['Return'], rf_pred, 'Random Forest')
# print('Best params:', gs.best_params_)
# print(f"RF RMSE = {rf_m['RMSE']:.5f}")


## 9. Feature Importance

In [ ]:
# Load canonical feature importances from CSV
fi = pd.read_csv(OUTPUT_DIR / 'Random_Forest_Feature_Importance.csv')
print(fi.to_string(index=False))


 Rank           Feature  Importance Score
    1      Market_Return          0.568669
    2          Momentum          0.226054
    3        Volatility          0.074775
    4     Lagged_Return          0.067019
    5        Financials          0.032729
    6     Risk_Free_Rate          0.018869
    7        Technology          0.006638
    8          Consumer          0.004870
    9        Healthcare          0.000378


In [ ]:
# Feature importance horizontal bar chart
plot = fi.sort_values('Importance Score', ascending=True)
colours_fi = plt.cm.viridis(np.linspace(0.2, 0.85, len(plot)))
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(plot['Feature'], plot['Importance Score'], color=colours_fi, edgecolor='white')
for i, (val, feat) in enumerate(zip(plot['Importance Score'], plot['Feature'])):
    ax.text(val+0.003, i, f'{val:.4f}', va='center', fontsize=8.5)
ax.set_title('Random Forest Feature Importance (MDI)', fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()


## 10. Model Comparison

In [ ]:
comparison = pd.read_csv(OUTPUT_DIR / 'Model_Comparison.csv')
print(comparison[['Model','RMSE','MAE','R2','N_Test']].to_string(index=False))

capm_rmse = comparison.loc[comparison['Model']=='CAPM','RMSE'].values[0]
ext_rmse  = comparison.loc[comparison['Model']=='Extended Model','RMSE'].values[0]
rf_rmse   = comparison.loc[comparison['Model']=='Random Forest','RMSE'].values[0]
print(f"\nExtended Model improvement over CAPM: {(capm_rmse-ext_rmse)/capm_rmse*100:.2f}%")
print(f"Random Forest improvement over CAPM:   {(capm_rmse-rf_rmse)/capm_rmse*100:.2f}%")
print(f"Random Forest improvement over Ext:    {(ext_rmse-rf_rmse)/ext_rmse*100:.2f}%")


          Model     RMSE      MAE        R2  N_Test
  Random Forest  0.06260  0.04789  0.363845    1440
 Extended Model  0.06364  0.04857  0.342402    1440
           CAPM  0.06403  0.04878  0.334350    1440

Extended Model improvement over CAPM: 0.61%
Random Forest improvement over CAPM:  2.20%
Random Forest improvement over Ext:   1.64%


In [ ]:
# RMSE bar chart with consistent colour scheme
plot = comparison.sort_values('RMSE')
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(plot['Model'], plot['RMSE'],
              color=[COLOURS.get(m,'#888') for m in plot['Model']],
              edgecolor='white', linewidth=0.8)
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.0002,
            f'{bar.get_height():.5f}', ha='center', va='bottom', fontsize=9)
ax.set_title('RMSE by Model (Out-of-Sample 2019–2024)', fontweight='bold')
ax.set_xlabel('Model'); ax.set_ylabel('RMSE')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()


## 11. Statistical Tests

In [ ]:
stat_tests = pd.read_csv(OUTPUT_DIR / 'Statistical_Tests.csv')
display_cols = ['Comparison','Statistic','p_Value','N']
print(stat_tests[display_cols].to_string(index=False))

print()
for _, row in stat_tests.iterrows():
    sig = '** (5%)' if row['p_Value'] < 0.05 else '* (10%)' if row['p_Value'] < 0.10 else 'ns'
    print(f"{row['Comparison']}: p = {row['p_Value']:.4f} → {sig}")


                          Comparison  Statistic   p_Value     N
             CAPM vs Extended Model   2.450829  0.014371  1440
             CAPM vs Random Forest    2.162340  0.030756  1440
   Extended Model vs Random Forest    1.661689  0.096793  1440

CAPM vs Extended Model: p = 0.0144 → ** (5%)
CAPM vs Random Forest: p = 0.0308 → ** (5%)
Extended Model vs Random Forest: p = 0.0968 → * (10%)


## 12. Sector Analysis

In [ ]:
sector = pd.read_csv(OUTPUT_DIR / 'Sector_Analysis.csv')
print(sector.to_string(index=False))

print()
best_by_sector = sector.loc[sector.groupby('Sector')['RMSE'].idxmin()]
print('Best model per sector:')
print(best_by_sector[['Sector','Model','RMSE']].to_string(index=False))


       Sector          Model      RMSE       MAE  N_Test
   Technology           CAPM  0.074197  0.053372     360
   Technology Extended Model  0.073473  0.053287     360
   Technology  Random Forest  0.072621  0.052891     360
   Financials           CAPM  0.062316  0.050054     360
   Financials Extended Model  0.062700  0.050470     360
   Financials  Random Forest  0.066715  0.053693     360
   Healthcare           CAPM  0.066738  0.052488     360
   Healthcare Extended Model  0.066126  0.051859     360
   Healthcare  Random Forest  0.062415  0.049016     360
     Consumer           CAPM  0.050582  0.039206     360
     Consumer Extended Model  0.050004  0.038666     360
     Consumer  Random Forest  0.045321  0.035970     360

Best model per sector:
    Sector          Model      RMSE
  Consumer  Random Forest  0.045321
Financials           CAPM  0.062316
Healthcare  Random Forest  0.062415
Technology  Random Forest  0.072621


In [ ]:
# Sector RMSE grouped bar chart
fig, ax = plt.subplots(figsize=(10, 5.5))
palette = [COLOURS['CAPM'], COLOURS['Extended Model'], COLOURS['Random Forest']]
sns.barplot(data=sector, x='Sector', y='RMSE', hue='Model', ax=ax,
            palette=palette, edgecolor='white')
ax.set_title('RMSE by Sector and Model', fontweight='bold')
ax.set_xlabel('Sector'); ax.set_ylabel('RMSE')
ax.legend(title='Model', frameon=True)
plt.tight_layout()
plt.show()


## 13. Actual vs Predicted Returns

In [ ]:
predictions = pd.read_csv(OUTPUT_DIR / 'Model_Predictions.csv', parse_dates=['Date'])

fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharex=True)
model_cols = [('CAPM','CAPM_Prediction'), ('Extended Model','Extended Model_Prediction'),
              ('Random Forest','Random Forest_Prediction')]

for ax, (model_name, col) in zip(axes, model_cols):
    plot = (predictions[['Date','Actual_Return',col]].dropna()
            .groupby('Date', as_index=False).mean(numeric_only=True))
    ax.plot(plot['Date'], plot['Actual_Return'], label='Actual', color='#2d2d2d', lw=1.8)
    ax.plot(plot['Date'], plot[col], label=f'{model_name} Predicted',
            color=COLOURS[model_name], lw=1.8)
    ax.axhline(0, color='black', lw=0.6, ls='--')
    ax.set_title(f'Actual vs Predicted — {model_name}', fontweight='bold')
    ax.set_ylabel('Monthly Return')
    ax.legend(frameon=True)

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()


## 14. Conclusion

| Finding | Detail |
|---------|--------|
| Best overall model | **Random Forest** (RMSE = 0.06260) |
| RF vs CAPM improvement | **2.20%** lower RMSE (p = 0.031, **) |
| Extended vs CAPM | **0.61%** lower RMSE (p = 0.014, **) |
| RF vs Extended | **1.64%** lower RMSE (p = 0.097, * 10% only) |
| Best sector (RF) | Consumer, Healthcare, Technology |
| Best sector (CAPM) | **Financials** — systemic risk dominates |
| Top RF feature | Market Return (56.9% MDI importance) |

**Conclusion:** The inclusion of additional financial information (lagged returns, volatility, momentum, sector indicators) does improve stock return prediction accuracy beyond the CAPM benchmark. The gains are modest in absolute terms — consistent with the inherent unpredictability of monthly equity returns — but statistically significant at conventional levels.
